# AquaTrack: ViTPose-B vs our YOLO swim model

Runs both models on the same swim photos and video, side by side. Nothing is trained.

- **Left:** our YOLO model from the SwimXYZ training (`swim_full/best.pt` in Drive).
- **Right:** ViTPose-B, off the shelf, trained on COCO people only.
- ViTPose only finds joints inside a box you give it, so it gets the swimmer's box from our YOLO model.
  Both panels show the same swimmer, and the comparison is only about **where the joints land**.
- Colors: **blue** = swimmer's left side, **orange** = right side, **green** = head and torso.
  Only joints the model is fairly sure about (score above 0.3) are drawn.

**Steps:** any runtime works (a T4 GPU makes the video faster).
1. Upload your photos and video with the folder icon on the left.
2. Run the cells from top to bottom. Change the file names in cells 2 to 9 if yours differ.
3. Results are also saved to Drive in `MyDrive/aquatrack_swim_full/compare/`.

**Part 2 (cells 4 and 5): was the SwimXYZ data the problem?** Compares the original `yolo11n-pose`
(trained on real COCO people only) with our model (the same yolo11n, fine-tuned on SwimXYZ's
computer-generated swimmers). Same model size, so any difference comes from the SwimXYZ training.
Here each model finds the swimmer on its own.

**Part 3 (cells 6 and 7): ViTPose-B vs MediaPipe.** MediaPipe gives 33 points (hands, heels and toes too).

**Part 4 (cells 8 and 9): hands and feet.** Whole-body models (133 points: body, feet, hands) in our YOLO's
swimmer box: ViTPose-B whole-body and RTMW-l whole-body, next to ViTPose-B (17 points) for reference.
To run only one part: cells 0 and 1, then that part's cells.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics==8.4.173
!pip install -q -U transformers

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/aquatrack_swim_full')
YOLO_MODEL = DRIVE / 'runs' / 'swim_full' / 'weights' / 'best.pt'
VITPOSE_ID = 'usyd-community/vitpose-base-simple'   # ViTPose-B, COCO 17 joints
OUT = DRIVE / 'compare'
OUT.mkdir(parents=True, exist_ok=True)
print('YOLO model found:', YOLO_MODEL.exists())

## 1. Load both models

In [ ]:
import subprocess
from pathlib import Path
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Video, display
from tqdm.auto import tqdm
from ultralytics import YOLO
from transformers import AutoProcessor, VitPoseForPoseEstimation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(str(YOLO_MODEL))
vp_proc = AutoProcessor.from_pretrained(VITPOSE_ID)
vitpose = VitPoseForPoseEstimation.from_pretrained(VITPOSE_ID).to(device).eval()
print('Running on', device)

SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16), (0, 1), (0, 2), (1, 3), (2, 4)]
LEFT, RIGHT = {1, 3, 5, 7, 9, 11, 13, 15}, {2, 4, 6, 8, 10, 12, 14, 16}
BLUE, ORANGE, GREEN = (255, 128, 0), (0, 140, 255), (0, 200, 0)   # BGR
MIN_SCORE = 0.3


def find(name):
    """Uploads can land in /content or in the top-level folder, so look in both."""
    for p in (Path(name), Path('/content') / name, Path('/') / name):
        if p.is_file():
            return p
    raise FileNotFoundError(f'{name} not found. Upload it with the folder icon on the left.')


def side_color(*joints):
    if set(joints) <= LEFT:
        return BLUE
    if set(joints) <= RIGHT:
        return ORANGE
    return GREEN


def draw(img, title, box=None, xy=None, scores=None):
    out = img.copy()
    s = max(1, round(out.shape[1] / 640))
    if box is not None:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (200, 200, 200), s)
    if xy is not None:
        ok = scores > MIN_SCORE
        for a, b in SKELETON:
            if ok[a] and ok[b]:
                pa, pb = tuple(map(int, xy[a])), tuple(map(int, xy[b]))
                cv2.line(out, pa, pb, side_color(a, b), 2 * s, cv2.LINE_AA)
        for j in range(17):
            if ok[j]:
                cv2.circle(out, tuple(map(int, xy[j])), 3 * s, side_color(j), -1, cv2.LINE_AA)
    label = title if xy is not None else title + ': no swimmer'
    cv2.putText(out, label, (10 * s, 30 * s), cv2.FONT_HERSHEY_SIMPLEX, 0.8 * s, (0, 0, 0), 4 * s, cv2.LINE_AA)
    cv2.putText(out, label, (10 * s, 30 * s), cv2.FONT_HERSHEY_SIMPLEX, 0.8 * s, (255, 255, 255), 2 * s, cv2.LINE_AA)
    return out


def run_yolo(img):
    """Most confident swimmer from our YOLO model: box (x1, y1, x2, y2), joints (17, 2), scores (17,)."""
    r = yolo.predict(img, conf=0.25, verbose=False)[0]
    if len(r.boxes) == 0:
        return None, None, None
    i = int(r.boxes.conf.argmax())
    box = r.boxes.xyxy[i].cpu().numpy()
    xy = r.keypoints.xy[i].cpu().numpy()
    scores = r.keypoints.conf[i].cpu().numpy() if r.keypoints.conf is not None else np.ones(17)
    return box, xy, scores


@torch.no_grad()
def run_vitpose(img, box):
    """ViTPose-B joints inside the given box."""
    x1, y1, x2, y2 = box
    coco_box = np.array([[x1, y1, x2 - x1, y2 - y1]], dtype=np.float32)   # ViTPose wants x, y, width, height
    pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    inputs = vp_proc(pil, boxes=[coco_box], return_tensors='pt').to(device)
    outputs = vitpose(**inputs)
    person = vp_proc.post_process_pose_estimation(outputs, boxes=[coco_box])[0][0]
    return person['keypoints'].cpu().numpy(), person['scores'].cpu().numpy()


def compare(img):
    """Returns (left panel, right panel, swimmer found)."""
    box, yxy, ysc = run_yolo(img)
    left = draw(img, 'Our YOLO', box, yxy, ysc)
    if box is None:
        return left, draw(img, 'ViTPose-B'), False
    vxy, vsc = run_vitpose(img, box)
    return left, draw(img, 'ViTPose-B', box, vxy, vsc), True

## 2. Photos

In [ ]:
PHOTOS = ['person_swimming2.jpeg', 'swimming.jpg']   # change to your file names

for name in PHOTOS:
    img = cv2.imread(str(find(name)))
    left, right, found = compare(img)
    both = np.hstack([left, right])
    cv2.imwrite(str(OUT / f'{Path(name).stem}_compare.jpg'), both)
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(both, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis('off')
    plt.show()

## 3. Video

In [ ]:
VIDEO = 'swim_underwater.mp4'   # change to your file name
PANEL_WIDTH = 960                # each side is shrunk to this width at most

src = find(VIDEO)
cap = cv2.VideoCapture(str(src))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
raw = Path('compare_raw.mp4')   # temporary, in the notebook's folder
final = OUT / f'{src.stem}_compare.mp4'
writer, found, frames = None, 0, 0
bar = tqdm(total=total, desc=src.name)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    h, w = frame.shape[:2]
    if w > PANEL_WIDTH:
        frame = cv2.resize(frame, (PANEL_WIDTH, round(h * PANEL_WIDTH / w)))
    h, w = frame.shape[:2]
    frame = frame[: h - h % 2, : w - w % 2]   # the video encoder needs even sizes
    left, right, hit = compare(frame)
    both = np.hstack([left, right])
    if writer is None:
        writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*'mp4v'), fps, (both.shape[1], both.shape[0]))
    writer.write(both)
    frames += 1
    found += hit
    bar.update(1)
bar.close()
cap.release()
writer.release()

# re-encode so the video plays in the browser
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw), '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p', str(final)], check=True)
raw.unlink()
print(f'Swimmer found in {found / frames:.0%} of {frames} frames (both panels use the same box).')
print('Saved to', final)
display(Video(str(final), embed=True, width=1000))

## 4. Original yolo11n-pose vs our SwimXYZ model: photos

Each panel shows the model's most confident person and that confidence in brackets, even when it is low,
so you can compare the joints. The app only counts a swimmer as found at 0.25 or above.

In [ ]:
PHOTOS = ['person_swimming2.jpeg', 'swimming.jpg']   # change to your file names
YOLOS = {
    'Original yolo11n': YOLO('yolo11n-pose.pt'),   # real COCO people, no swimmers
    'Our yolo11n (SwimXYZ)': yolo,                   # the same model fine-tuned on SwimXYZ
    # 'Original yolo11x': YOLO('yolo11x-pose.pt'),   # remove the # to see if a bigger model alone helps
}
FOUND_CONF = 0.25   # the app's threshold for "swimmer found"


def best_person(model, img):
    """(confidence, box, joints, joint scores) of the most confident person, or None."""
    r = model.predict(img, conf=0.01, verbose=False)[0]
    if len(r.boxes) == 0:
        return None
    i = int(r.boxes.conf.argmax())
    scores = r.keypoints.conf[i].cpu().numpy() if r.keypoints.conf is not None else np.ones(17)
    return float(r.boxes.conf[i]), r.boxes.xyxy[i].cpu().numpy(), r.keypoints.xy[i].cpu().numpy(), scores


def yolo_panels(img):
    """One drawn panel per model, plus whether each found the swimmer at FOUND_CONF."""
    panels, found = [], []
    for name, model in YOLOS.items():
        person = best_person(model, img)
        if person is None:
            panels.append(draw(img, name))
            found.append(False)
            continue
        conf, box, xy, scores = person
        panels.append(draw(img, f'{name} ({conf:.2f})', box, xy, scores))
        found.append(conf >= FOUND_CONF)
    return panels, found


for name in PHOTOS:
    img = cv2.imread(str(find(name)))
    panels, _ = yolo_panels(img)
    both = np.hstack(panels)
    cv2.imwrite(str(OUT / f'{Path(name).stem}_yolo11n_vs_ours.jpg'), both)
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(both, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis('off')
    plt.show()

## 5. Original yolo11n-pose vs our SwimXYZ model: video

In [ ]:
VIDEO = 'swim_underwater.mp4'   # change to your file name
PANEL_WIDTH = 960

src = find(VIDEO)
cap = cv2.VideoCapture(str(src))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
raw = Path('compare_raw.mp4')   # temporary, in the notebook's folder
final = OUT / f'{src.stem}_yolo11n_vs_ours.mp4'
writer, frames = None, 0
found = [0] * len(YOLOS)
bar = tqdm(total=total, desc=src.name)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    h, w = frame.shape[:2]
    if w > PANEL_WIDTH:
        frame = cv2.resize(frame, (PANEL_WIDTH, round(h * PANEL_WIDTH / w)))
    h, w = frame.shape[:2]
    frame = frame[: h - h % 2, : w - w % 2]   # the video encoder needs even sizes
    panels, hits = yolo_panels(frame)
    both = np.hstack(panels)
    if writer is None:
        writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*'mp4v'), fps, (both.shape[1], both.shape[0]))
    writer.write(both)
    frames += 1
    found = [f + hit for f, hit in zip(found, hits)]
    bar.update(1)
bar.close()
cap.release()
writer.release()

subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw), '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p', str(final)], check=True)
raw.unlink()
for name, f in zip(YOLOS, found):
    print(f'{name}: swimmer found in {f / frames:.0%} of {frames} frames')
print('Saved to', final)
display(Video(str(final), embed=True, width=1000))

## 6. ViTPose-B vs MediaPipe: photos

**Left:** ViTPose-B in our YOLO's swimmer box, as in cells 2 and 3. **Right:** MediaPipe Pose (heavy model),
which finds the person on its own and gives **33 points**: the same body joints plus extra face points, heels,
toes, and three points per hand (thumb, index and pinky).
Filled dots are points MediaPipe thinks are visible. Hollow dots are its guess for points it thinks are hidden.
If MediaPipe misses swimmers, set `MP_CROP_TO_YOLO_BOX = True` to give it only the area around our YOLO's box.

In [ ]:
!pip install -q mediapipe
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

PHOTOS = ['person_swimming2.jpeg', 'swimming.jpg']   # change to your file names
MP_CROP_TO_YOLO_BOX = False   # True: MediaPipe only sees the area around our YOLO's swimmer box

MP_MODEL = Path('pose_landmarker_heavy.task')
if not MP_MODEL.exists():
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
                               'pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task', MP_MODEL)

# MediaPipe's 33 points: 0 nose, 1-10 eyes/ears/mouth, 11-16 shoulders/elbows/wrists,
# 17-22 pinky/index/thumb, 23-28 hips/knees/ankles, 29-30 heels, 31-32 toes. Odd = left from 7 on.
MP_EDGES = [(0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10), (11, 12), (11, 13),
            (13, 15), (15, 17), (15, 19), (15, 21), (17, 19), (12, 14), (14, 16), (16, 18), (16, 20),
            (16, 22), (18, 20), (11, 23), (12, 24), (23, 24), (23, 25), (24, 26), (25, 27), (26, 28),
            (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32)]
MP_LEFT = {1, 2, 3} | set(range(7, 33, 2))
MP_RIGHT = {4, 5, 6} | set(range(8, 33, 2))


def mp_landmarker(video=False):
    """A MediaPipe pose landmarker; video mode follows the swimmer from frame to frame."""
    mode = mp_vision.RunningMode.VIDEO if video and not MP_CROP_TO_YOLO_BOX else mp_vision.RunningMode.IMAGE
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(MP_MODEL)), running_mode=mode, num_poses=1)
    return mp_vision.PoseLandmarker.create_from_options(options)


def run_mediapipe(landmarker, img, timestamp_ms=None):
    """MediaPipe's 33 points in pixels (33, 2) and their visibility (33,), or (None, None)."""
    x0, y0, crop = 0, 0, img
    if MP_CROP_TO_YOLO_BOX:
        box, _, _ = run_yolo(img)
        if box is None:
            return None, None
        x1, y1, x2, y2 = box
        pad_x, pad_y = 0.15 * (x2 - x1), 0.15 * (y2 - y1)
        x0, y0 = int(max(0, x1 - pad_x)), int(max(0, y1 - pad_y))
        crop = img[y0:int(min(img.shape[0], y2 + pad_y)), x0:int(min(img.shape[1], x2 + pad_x))]
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
    if timestamp_ms is None or MP_CROP_TO_YOLO_BOX:   # same rule as mp_landmarker's mode
        result = landmarker.detect(image)
    else:
        result = landmarker.detect_for_video(image, timestamp_ms)
    if not result.pose_landmarks:
        return None, None
    h, w = crop.shape[:2]
    points = result.pose_landmarks[0]
    xy = np.array([[p.x * w + x0, p.y * h + y0] for p in points])
    vis = np.array([1.0 if p.visibility is None else p.visibility for p in points])
    return xy, vis


def mp_color(*points):
    if set(points) <= MP_LEFT:
        return BLUE
    if set(points) <= MP_RIGHT:
        return ORANGE
    return GREEN


def draw_mp(img, title, xy=None, vis=None):
    out = img.copy()
    s = max(1, round(out.shape[1] / 640))
    if xy is not None:
        for a, b in MP_EDGES:
            cv2.line(out, tuple(map(int, xy[a])), tuple(map(int, xy[b])), mp_color(a, b), 2 * s, cv2.LINE_AA)
        for j in range(33):
            filled = -1 if vis[j] >= MIN_SCORE else max(1, s)
            cv2.circle(out, tuple(map(int, xy[j])), 3 * s, mp_color(j), filled, cv2.LINE_AA)
    label = title if xy is not None else title + ': no swimmer'
    cv2.putText(out, label, (10 * s, 30 * s), cv2.FONT_HERSHEY_SIMPLEX, 0.8 * s, (0, 0, 0), 4 * s, cv2.LINE_AA)
    cv2.putText(out, label, (10 * s, 30 * s), cv2.FONT_HERSHEY_SIMPLEX, 0.8 * s, (255, 255, 255), 2 * s, cv2.LINE_AA)
    return out


def vitpose_vs_mediapipe(img, landmarker, timestamp_ms=None):
    """(ViTPose panel, MediaPipe panel, ViTPose found, MediaPipe found)"""
    _, vit_panel, vit_found = compare(img)
    xy, vis = run_mediapipe(landmarker, img, timestamp_ms)
    return vit_panel, draw_mp(img, 'MediaPipe (33 points)', xy, vis), vit_found, xy is not None


photo_landmarker = mp_landmarker()
for name in PHOTOS:
    img = cv2.imread(str(find(name)))
    vit_panel, mp_panel, _, _ = vitpose_vs_mediapipe(img, photo_landmarker)
    both = np.hstack([vit_panel, mp_panel])
    cv2.imwrite(str(OUT / f'{Path(name).stem}_vitpose_vs_mediapipe.jpg'), both)
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(both, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis('off')
    plt.show()

## 7. ViTPose-B vs MediaPipe: video

In [ ]:
VIDEO = 'swim_underwater.mp4'   # change to your file name
PANEL_WIDTH = 960

src = find(VIDEO)
cap = cv2.VideoCapture(str(src))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
raw = Path('compare_raw.mp4')   # temporary, in the notebook's folder
final = OUT / f'{src.stem}_vitpose_vs_mediapipe.mp4'
video_landmarker = mp_landmarker(video=True)
writer, frames, vit_found, mp_found = None, 0, 0, 0
bar = tqdm(total=total, desc=src.name)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    h, w = frame.shape[:2]
    if w > PANEL_WIDTH:
        frame = cv2.resize(frame, (PANEL_WIDTH, round(h * PANEL_WIDTH / w)))
    h, w = frame.shape[:2]
    frame = frame[: h - h % 2, : w - w % 2]   # the video encoder needs even sizes
    vit_panel, mp_panel, vit_hit, mp_hit = vitpose_vs_mediapipe(frame, video_landmarker, int(frames * 1000 / fps))
    both = np.hstack([vit_panel, mp_panel])
    if writer is None:
        writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*'mp4v'), fps, (both.shape[1], both.shape[0]))
    writer.write(both)
    frames += 1
    vit_found += vit_hit
    mp_found += mp_hit
    bar.update(1)
bar.close()
cap.release()
writer.release()
video_landmarker.close()

subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw), '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p', str(final)], check=True)
raw.unlink()
print(f'ViTPose (in our YOLO box): swimmer found in {vit_found / frames:.0%} of {frames} frames')
print(f'MediaPipe: swimmer found in {mp_found / frames:.0%} of {frames} frames')
print('Saved to', final)
display(Video(str(final), embed=True, width=1000))

## 8. Hands and feet: whole-body models, photos

Whole-body models give **133 points**: the 17 body joints, 3 points per foot (big toe, small toe, heel),
21 points per hand and 68 face points. Face points are not drawn here.
All panels use our YOLO's swimmer box, so the comparison is only about where the points land.

- **ViTPose-B (17 points):** the model from cells 2 and 3, for reference. Set `SHOW_BODY_ONLY = False` to hide it.
- **ViTPose-B whole-body:** the same ViTPose size, trained for whole-body points.
- **RTMW-l whole-body:** a newer whole-body model with higher whole-body scores than ViTPose (70.1 AP at
  384x288, almost the same as the biggest RTMW-x at 70.2, which has no ONNX download).

Hand points are drawn smaller. Both models run on the CPU through rtmlib, so the video in cell 9 takes a few
minutes. The models download on first run (about 360 MB for ViTPose-B whole-body).
To try another size, remove the `#` in front of a line in `WHOLEBODY`.

In [ ]:
!pip install -q onnxruntime
!pip install -q --no-deps rtmlib   # --no-deps keeps Colab's own OpenCV
from rtmlib import RTMPose, ViTPose

PHOTOS = ['person_swimming2.jpeg', 'swimming.jpg']   # change to your file names
SHOW_BODY_ONLY = True   # also show ViTPose-B (17 points) from cells 2 and 3
EASY_VITPOSE = 'https://huggingface.co/JunkyByte/easy_ViTPose/resolve/main/onnx/wholebody/'
RTMW = 'https://download.openmmlab.com/mmpose/v1/projects/rtmw/onnx_sdk/'
WHOLEBODY = {   # name: (kind, model file, input size as width, height)
    'ViTPose-B whole-body': ('vitpose', EASY_VITPOSE + 'vitpose-b-wholebody.onnx', (192, 256)),
    'RTMW-l whole-body': ('rtmw', RTMW + 'rtmw-dw-x-l_simcc-cocktail14_270e-384x288_20231122.zip', (288, 384)),
    # 'ViTPose-L whole-body': ('vitpose', EASY_VITPOSE + 'vitpose-l-wholebody.onnx', (192, 256)),   # 1.2 GB
    # 'ViTPose-S whole-body': ('vitpose', EASY_VITPOSE + 'vitpose-s-wholebody.onnx', (192, 256)),   # small
    # 'RTMW-m whole-body': ('rtmw', RTMW + 'rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.zip', (192, 256)),   # small
    # The real RTMW-x, exported to Drive by C1 of aquatrack_rtmw_swim_finetune.ipynb (swim = after training):
    # 'RTMW-x stock': ('rtmw', '/content/drive/MyDrive/aquatrack_swim_full/rtmw/rtmw_x_stock.onnx', (288, 384)),
    # 'RTMW-x swim': ('rtmw', '/content/drive/MyDrive/aquatrack_swim_full/rtmw/rtmw_x_swim.onnx', (288, 384)),
}
WB_MODELS = {}
for name, (kind, model_file, size) in WHOLEBODY.items():
    cls = ViTPose if kind == 'vitpose' else RTMPose
    WB_MODELS[name] = cls(model_file, model_input_size=size, backend='onnxruntime', device='cpu')

# COCO-WholeBody's 133 points: 0-16 body (as in cell 1), 17-19 left foot (big toe, small toe, heel),
# 20-22 right foot, 23-90 face, 91-111 left hand, 112-132 right hand. Each hand: wrist, then 4 points
# per finger from thumb to pinky.
LEFT_FOOT, RIGHT_FOOT = [17, 18, 19], [20, 21, 22]
LEFT_HAND, RIGHT_HAND = list(range(91, 112)), list(range(112, 133))
FOOT_EDGES = [(15, 17), (15, 18), (15, 19), (16, 20), (16, 21), (16, 22)]
HAND_EDGES = [(9, 91), (10, 112)]
for wrist in (91, 112):
    for finger in range(5):
        start = wrist + 1 + 4 * finger
        HAND_EDGES += [(wrist, start), (start, start + 1), (start + 1, start + 2), (start + 2, start + 3)]
WB_LEFT = LEFT | set(LEFT_FOOT) | set(LEFT_HAND)
WB_RIGHT = RIGHT | set(RIGHT_FOOT) | set(RIGHT_HAND)


def wb_color(*points):
    if set(points) <= WB_LEFT:
        return BLUE
    if set(points) <= WB_RIGHT:
        return ORANGE
    return GREEN


def run_wholebody(model, img, box):
    """133 points in pixels (133, 2) and scores (133,) inside the box. Both models were trained on RGB."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    xy, scores = model(rgb, bboxes=[box.tolist()])
    return xy[0], scores[0]


def draw_wb(img, title, box=None, xy=None, scores=None):
    out = draw(img, title, box, xy[:17] if xy is not None else None, scores[:17] if xy is not None else None)
    if xy is None:
        return out
    s = max(1, round(out.shape[1] / 640))
    ok = scores > MIN_SCORE
    for edges, width in ((FOOT_EDGES, 2 * s), (HAND_EDGES, s)):
        for a, b in edges:
            if ok[a] and ok[b]:
                cv2.line(out, tuple(map(int, xy[a])), tuple(map(int, xy[b])), wb_color(a, b), width, cv2.LINE_AA)
    for points, radius in ((LEFT_FOOT + RIGHT_FOOT, 3 * s), (LEFT_HAND + RIGHT_HAND, 2 * s)):
        for j in points:
            if ok[j]:
                cv2.circle(out, tuple(map(int, xy[j])), radius, wb_color(j), -1, cv2.LINE_AA)
    return out


def seen(scores):
    """Which hands and feet the model was fairly sure about: most of their points above MIN_SCORE."""
    ok = scores > MIN_SCORE
    return {'left hand': ok[LEFT_HAND].sum() >= 11, 'right hand': ok[RIGHT_HAND].sum() >= 11,
            'left foot': ok[LEFT_FOOT].sum() >= 2, 'right foot': ok[RIGHT_FOOT].sum() >= 2}


def wholebody_panels(img):
    """One panel per model, plus {model name: seen(...)} (empty when no swimmer was found)."""
    box, _, _ = run_yolo(img)
    panels, parts = [], {}
    if SHOW_BODY_ONLY:
        if box is None:
            panels.append(draw(img, 'ViTPose-B (17 points)'))
        else:
            panels.append(draw(img, 'ViTPose-B (17 points)', box, *run_vitpose(img, box)))
    for name, model in WB_MODELS.items():
        if box is None:
            panels.append(draw(img, name))
            continue
        xy, scores = run_wholebody(model, img, box)
        panels.append(draw_wb(img, name, box, xy, scores))
        parts[name] = seen(scores)
    return panels, parts


for name in PHOTOS:
    img = cv2.imread(str(find(name)))
    panels, _ = wholebody_panels(img)
    both = np.hstack(panels)
    cv2.imwrite(str(OUT / f'{Path(name).stem}_wholebody.jpg'), both)
    plt.figure(figsize=(8 * len(panels), 6))
    plt.imshow(cv2.cvtColor(both, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis('off')
    plt.show()

## 9. Hands and feet: whole-body models, video

At the end it prints how often each model was fairly sure about each hand and foot. That only says how
sure the model was, not whether it was right, so watch the video to judge.

In [ ]:
VIDEO = 'swim_underwater.mp4'   # change to your file name
PANEL_WIDTH = 640                # smaller panels, since there are three side by side

src = find(VIDEO)
cap = cv2.VideoCapture(str(src))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
raw = Path('compare_raw.mp4')   # temporary, in the notebook's folder
final = OUT / f'{src.stem}_wholebody.mp4'
writer, frames = None, 0
counts = {name: {} for name in WB_MODELS}
bar = tqdm(total=total, desc=src.name)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    h, w = frame.shape[:2]
    if w > PANEL_WIDTH:
        frame = cv2.resize(frame, (PANEL_WIDTH, round(h * PANEL_WIDTH / w)))
    h, w = frame.shape[:2]
    frame = frame[: h - h % 2, : w - w % 2]   # the video encoder needs even sizes
    panels, parts = wholebody_panels(frame)
    both = np.hstack(panels)
    if writer is None:
        writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*'mp4v'), fps, (both.shape[1], both.shape[0]))
    writer.write(both)
    frames += 1
    for name, part_seen in parts.items():
        for part, hit in part_seen.items():
            counts[name][part] = counts[name].get(part, 0) + int(hit)
    bar.update(1)
bar.close()
cap.release()
writer.release()

subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw), '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p', str(final)], check=True)
raw.unlink()
for name, part_counts in counts.items():
    summary = ', '.join(f'{part} {n / frames:.0%}' for part, n in part_counts.items()) or 'no swimmer found'
    print(f'{name} ({frames} frames): {summary}')
print('Saved to', final)
display(Video(str(final), embed=True, width=1000))